<a href="https://colab.research.google.com/github/ksnny48985-ops/milaisw/blob/main/P5_%EC%99%B8%EC%B6%9C%EC%B6%94%EC%B2%9C_%EC%8B%9C%EC%9E%91%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗺️ 외출 추천 지도 — TOP 10 (Day 2)

> **위경도 + 거리 계산 + folium — 시작 코드 템플릿**

- ✏️ `# TODO:` 부분을 채워가며 진행
- 🎤 막힐 때 노트북에 적힌 **바이브 코딩 프롬프트** 활용
- ✅ 각 미션 끝에 자동 체크포인트

**오늘의 목표**
1. 시설 좌표 데이터 2~5개 다운로드
2. concat으로 통합 + lat/lon 정제
3. Haversine 거리 계산 + 정제
4. 점수화 → TOP 10 + folium 지도


## 🔧 0. 환경 셋업

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# folium은 Colab 기본 미설치
!pip install -q folium

In [4]:
pip install geopy

In [14]:
import requests
import json

def save_population_data_to_json(api_key, file_name="gyeryong_population.json"):
    # 1. API 엔드포인트 설정 [cite: 14, 20]
    url = "http://apis.data.go.kr/5580000/popltnRegistSttusService/getpopltnRegistSttus"

    # 2. 요청 파라미터 설정
    # 가이드에 따라 필수 항목(serviceKey, currentPage, perPage)을 포함합니다.
    params = {
        "serviceKey": api_key,  # 공공데이터포털에서 발급받은 인증키
        "currentPage": 1,        # 페이지 번호
        "perPage": 100,          # 한 페이지 결과 수 (최대치로 설정 권장)
        "SEXDSTN": "남",         # 성별 (필요 시 선택)
        # "SE": "두마면"         # 특정 지역구분이 필요할 경우 주석 해제
    }

    try:
        # 3. API 호출
        response = requests.get(url, params=params)

        # 상태 코드가 200(성공)인지 확인
        if response.status_code == 200:
            # 4. JSON 데이터 파싱
            data = response.json()

            # API 응답 결과가 정상(NORMAL_SERVICE)인지 확인 [cite: 25, 28]
            if data.get("header", {}).get("resultCode") == "00":
                # 5. 파일로 저장
                with open(file_name, 'w', encoding='utf-8') as f:
                    json.dump(data, f, ensure_ascii=False, indent=4)

                print(f"성공: 데이터를 '{file_name}'에 저장했습니다.")
                return data
            else:
                msg = data.get("header", {}).get("resultMsg")
                print(f"API 에러: {msg}")
        else:
            print(f"HTTP 에러: {response.status_code}")

    except Exception as e:
        print(f"오류 발생: {e}")

# --- 실행부 ---
# 여기에 발급받으신 API 키를 입력하세요.
MY_API_KEY = "f2c4d1800a1c5574e4179a966bb4d2efbef3f37dac9d24663bb0ea175c158219"
save_population_data_to_json(MY_API_KEY)

성공: 데이터를 'gyeryong_population.json'에 저장했습니다.


{'body': [{'DATE': 200901,
   'SE': '두마면',
   'SEXDSTN': '남',
   'POPLTN_CO': 3104,
   'HSHLD_CO': 2306,
   'POPLTN_IRDS': 136,
   'HSHLD_IRDS': 37,
   'ODSN_POPLTN': None},
  {'DATE': 200901,
   'SE': '엄사면',
   'SEXDSTN': '남',
   'POPLTN_CO': 8990,
   'HSHLD_CO': 6399,
   'POPLTN_IRDS': 20,
   'HSHLD_IRDS': 4,
   'ODSN_POPLTN': None},
  {'DATE': 200901,
   'SE': '신도안면',
   'SEXDSTN': '남',
   'POPLTN_CO': 4296,
   'HSHLD_CO': 2364,
   'POPLTN_IRDS': 139,
   'HSHLD_IRDS': 29,
   'ODSN_POPLTN': None},
  {'DATE': 200901,
   'SE': '금암동',
   'SEXDSTN': '남',
   'POPLTN_CO': 4163,
   'HSHLD_CO': 2750,
   'POPLTN_IRDS': 4,
   'HSHLD_IRDS': 6,
   'ODSN_POPLTN': None},
  {'DATE': 200902,
   'SE': '두마면',
   'SEXDSTN': '남',
   'POPLTN_CO': 3213,
   'HSHLD_CO': 2372,
   'POPLTN_IRDS': 234,
   'HSHLD_IRDS': 66,
   'ODSN_POPLTN': None},
  {'DATE': 200902,
   'SE': '엄사면',
   'SEXDSTN': '남',
   'POPLTN_CO': 8986,
   'HSHLD_CO': 6403,
   'POPLTN_IRDS': -39,
   'HSHLD_IRDS': 4,
   'ODSN_POPLTN': None},
 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os

pd.set_option('display.max_columns', None)
print('OK!')

In [ ]:
# TODO: 본인 데이터 폴더
DATA_DIR = '/content/drive/MyDrive/Day2_p5/'

print('파일 목록:')
for f in os.listdir(DATA_DIR):
    print(' -', f)

### 본인 부대 좌표 입력

> Google Maps에서 부대 위치 우클릭 → 좌표 복사 (위도, 경도 순서)


In [ ]:
# TODO: 본인 부대 좌표
BASE_LAT = 38.10   # 위도 (33~38)
BASE_LON = 128.20  # 경도 (124~132)

print(f'부대 좌표: ({BASE_LAT}, {BASE_LON})')

# 한국 범위 체크
assert 33 <= BASE_LAT <= 38, '위도가 한국 범위 밖'
assert 124 <= BASE_LON <= 132, '경도가 한국 범위 밖'

---
# 🎯 미션 1 — 좌표 데이터 다운로드 (35분)

위/경도 컬럼이 있는 시설 데이터를 2~5개 받아주세요.

| 카테고리 | 키워드 | 파일명 |
|---|---|---|
| 도서관 | 전국 도서관 표준 데이터 | library.csv |
| 문화시설 | 박물관 / 영화관 | culture.csv |
| 음식점 | 일반음식점 | food.csv |
| 카페 | 휴게음식점 (커피) | cafe.csv |
| 관광지 | 관광지 / 공원 | tour.csv |

> ✅ 0번 셀에서 2~5개 파일이 출력되면 완료


---
# 🎯 미션 2 — 좌표 통합 DataFrame (60분)


## 2-1. 도서관 (예시)

In [ ]:
# TODO: 본인 파일명 / 컬럼명에 맞게 수정
df_lib = pd.read_csv(DATA_DIR + 'library.csv', encoding='cp949')
print('shape:', df_lib.shape)
print('컬럼:', list(df_lib.columns)[:10])
df_lib.head(2)

In [ ]:
# 컬럼 통일: 이름, lat, lon, 종류
# TODO: 위/경도 컬럼명을 확인하고 수정
df_lib = df_lib.rename(columns={
    '도서관명': '이름',
    '위도': 'lat',
    '경도': 'lon',
})
df_lib['종류'] = '도서관'

# 숫자 변환
df_lib['lat'] = pd.to_numeric(df_lib['lat'], errors='coerce')
df_lib['lon'] = pd.to_numeric(df_lib['lon'], errors='coerce')

df_lib[['이름', 'lat', 'lon', '종류']].head()

## 2-2. 카페 / 음식점 / 기타도 동일하게

In [ ]:
# TODO: 본인이 받은 파일들에 동일한 패턴으로 적용

# 카페
df_cafe = pd.read_csv(DATA_DIR + 'cafe.csv', encoding='cp949')
df_cafe = df_cafe.rename(columns={'상호명': '이름', '위도': 'lat', '경도': 'lon'})
df_cafe['종류'] = '카페'
df_cafe['lat'] = pd.to_numeric(df_cafe['lat'], errors='coerce')
df_cafe['lon'] = pd.to_numeric(df_cafe['lon'], errors='coerce')

# 음식점
df_food = pd.read_csv(DATA_DIR + 'food.csv', encoding='cp949')
df_food = df_food.rename(columns={'사업장명': '이름', '위도': 'lat', '경도': 'lon'})
df_food['종류'] = '음식점'
df_food['lat'] = pd.to_numeric(df_food['lat'], errors='coerce')
df_food['lon'] = pd.to_numeric(df_food['lon'], errors='coerce')

# 더 받았으면 동일하게 추가

## 2-3. 합치기 (concat)

In [ ]:
common = ['이름', 'lat', 'lon', '종류']

all_places = pd.concat([
    df_lib[common],
    df_cafe[common],
    df_food[common],
    # 더 있으면 추가
], ignore_index=True)

print('전체:', all_places.shape)
print('종류 분포:')
print(all_places['종류'].value_counts())

## 2-4. 한국 범위 검증

In [ ]:
# 좌표 결측 제거
before = len(all_places)
all_places = all_places.dropna(subset=['lat', 'lon'])
print(f'결측 제거: {before} → {len(all_places)}')

# 한국 범위 안에 있는 행만
in_korea = (all_places['lat'].between(33, 38)) & (all_places['lon'].between(124, 132))
print(f'한국 범위: {in_korea.sum()} / {len(all_places)}')

all_places = all_places[in_korea].reset_index(drop=True)
all_places.head()

## 2-5. ✅ 체크포인트

In [ ]:
checks = {
    'all_places 존재': 'all_places' in dir(),
    'lat/lon이 float': all_places[['lat', 'lon']].dtypes.apply(lambda d: d.kind == 'f').all(),
    '한국 범위 안': all_places['lat'].between(33, 38).all() and all_places['lon'].between(124, 132).all(),
    '종류 2가지 이상': all_places['종류'].nunique() >= 2,
}
for k, v in checks.items():
    print(f'{"✅" if v else "❌"} {k}')

if all(checks.values()):
    print('\n🎉 미션 2 완료!')

---
# 🎯 미션 3 — 거리 계산 + 정제 (60분)


## 3-1. Haversine 함수 정의

지구의 곡률을 고려한 두 좌표 사이의 거리(km)를 계산하는 공식입니다.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """두 좌표 사이의 거리 (km)"""
    R = 6371  # 지구 반지름
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (np.sin(dlat/2)**2 +
         np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(a))

# 테스트 (서울 ~ 부산 = 약 325km)
test = haversine(37.5665, 126.9780, 35.1796, 129.0756)
print(f'서울-부산: {test:.1f} km (정상이면 320~330)')

## 3-2. 부대로부터의 거리 계산 (벡터화)

In [ ]:
# 모든 시설에 대해 한 번에
all_places['거리_km'] = haversine(
    BASE_LAT, BASE_LON,
    all_places['lat'], all_places['lon']
)

print('거리 분포:')
print(all_places['거리_km'].describe())

## 3-3. 외출 가능 거리로 필터링

In [ ]:
# TODO: 외출 가능 거리 (본인 환경에 맞게)
MAX_DIST = 5  # km

nearby = all_places[all_places['거리_km'] <= MAX_DIST].copy()
print(f'{MAX_DIST}km 이내: {len(nearby)}개')

print()
print('종류별 분포:')
print(nearby['종류'].value_counts())

## 3-4. 종류 통일 + 중복 제거

In [ ]:
# 큰 카테고리 매핑 (선택)
category_map = {
    '도서관': '문화·학습',
    '박물관': '문화·학습',
    '카페':   '먹거리',
    '음식점': '먹거리',
    '관광지': '여가',
    '공원':   '여가',
}
nearby['대분류'] = nearby['종류'].map(category_map).fillna('기타')

# 중복 제거 (같은 이름 + 같은 좌표)
before = len(nearby)
nearby = nearby.drop_duplicates(subset=['이름', 'lat', 'lon']).reset_index(drop=True)
print(f'중복 제거: {before} → {len(nearby)}')

nearby.head()

## 3-5. 저장 + ✅ 체크포인트

In [ ]:
nearby.to_csv(DATA_DIR + 'nearby.csv', encoding='utf-8-sig', index=False)

checks = {
    '거리 컬럼 존재': '거리_km' in nearby.columns,
    f'{MAX_DIST}km 이내 N개': len(nearby) > 0,
    '대분류 컬럼': '대분류' in nearby.columns,
    'nearby.csv 저장': os.path.exists(DATA_DIR + 'nearby.csv'),
}
for k, v in checks.items():
    print(f'{"✅" if v else "❌"} {k}')

---
# 🎯 미션 4 — TOP 10 + folium 지도 (60분)


## 4-1. 거리 점수 + 종류 가중치

In [ ]:
# 거리 점수 (가까울수록 100점)
nearby['거리_점수'] = (MAX_DIST - nearby['거리_km']) / MAX_DIST * 100

# TODO: 본인 취향 가중치 (합 1.0이면 더 명확)
TYPE_WEIGHTS = {
    '카페':   1.0,
    '도서관': 0.8,
    '음식점': 0.7,
    '박물관': 0.6,
    '관광지': 0.5,
    '공원':   0.5,
}

# 매핑 (없는 종류는 기본값 0.5)
nearby['종류_가중치'] = nearby['종류'].map(TYPE_WEIGHTS).fillna(0.5)

# 총점
nearby['총점'] = nearby['거리_점수'] * nearby['종류_가중치']

nearby[['이름', '종류', '거리_km', '거리_점수', '종류_가중치', '총점']].head()

## 4-2. TOP 10 추출

In [ ]:
top10 = nearby.sort_values('총점', ascending=False).head(10).reset_index(drop=True)

print('🏆 외출 추천 TOP 10:')
print()
for i, row in top10.iterrows():
    print(f"{i+1:2}. {row['이름']:30} ({row['종류']}, {row['거리_km']:.2f}km) 총점 {row['총점']:.1f}")

## 4-3. folium 지도

지도가 마지막 셀에서 `m`만 적으면 인터랙티브 지도로 표시됩니다.


In [ ]:
# 부대 중심 지도
m = folium.Map(location=[BASE_LAT, BASE_LON], zoom_start=14)

# 부대 마커
folium.Marker(
    [BASE_LAT, BASE_LON],
    icon=folium.Icon(color='black', icon='star'),
    popup=folium.Popup('🪖 우리 부대', max_width=200)
).add_to(m)

# 1km 반경 원 (참고용)
folium.Circle(
    [BASE_LAT, BASE_LON],
    radius=1000,  # m 단위
    color='gray',
    fill=False,
    dash_array='5'
).add_to(m)

# TOP 10 마커 (amber 큰 원)
for _, row in top10.iterrows():
    popup_text = (
        f"<b>{row['이름']}</b><br>"
        f"{row['종류']} | {row['거리_km']:.2f}km<br>"
        f"총점 {row['총점']:.1f}"
    )
    folium.CircleMarker(
        [row['lat'], row['lon']],
        radius=10,
        color='#F4A261',
        fill=True,
        fill_color='#F4A261',
        fill_opacity=0.9,
        popup=folium.Popup(popup_text, max_width=200)
    ).add_to(m)

# 나머지 시설 (작은 teal 점)
rest = nearby[~nearby.index.isin(top10.index)]
for _, row in rest.iterrows():
    folium.CircleMarker(
        [row['lat'], row['lon']],
        radius=3,
        color='#1C7293',
        fill=True,
        fill_opacity=0.5,
        popup=f"{row['이름']} ({row['종류']})"
    ).add_to(m)

# HTML로 저장
m.save(DATA_DIR + 'recommendations.html')
print('✅ HTML 지도 저장됨')

m  # Colab에서 인터랙티브 지도 표시

## 4-4. 한 줄 인사이트

In [ ]:
top1 = top10.iloc[0]

# TODO: 발표용 한 줄
INSIGHT = (
    f"외출 추천 1등은 '{top1['이름']}' ({top1['종류']}). "
    f"{top1['거리_km']:.2f}km로 가깝고 본인 취향과 맞음."
)
print('=' * 60)
print(INSIGHT)
print('=' * 60)

## 4-5. ✅ 체크포인트

In [ ]:
checks = {
    'TYPE_WEIGHTS 정의': len(TYPE_WEIGHTS) > 0,
    '총점 컬럼': '총점' in nearby.columns,
    'TOP 10 추출': len(top10) == 10 or len(top10) == len(nearby),
    'folium 지도 존재': 'm' in dir(),
    'HTML 저장됨': os.path.exists(DATA_DIR + 'recommendations.html'),
}
for k, v in checks.items():
    print(f'{"✅" if v else "❌"} {k}')

if all(checks.values()):
    print('\n🎉 모든 미션 완료!')

---
# 📢 발표 준비 (5분)

1. 본인 부대 + 본인 취향 (한 줄)
2. 어떤 데이터를 모았나
3. 가장 어려웠던 단계 + 해결법
4. TOP 10 표 + folium 지도
5. 1등 추천 한 줄


---
# ⭐ 도전 과제

## 도전 1 — 거리 계산 변형 (지수 감쇠)
멀어질수록 더 빨리 점수가 떨어지는 모델


In [ ]:
# 지수 감쇠 거리 점수
import numpy as np
nearby['거리_점수_exp'] = 100 * np.exp(-nearby['거리_km'] / 1.5)

# 비교
compare = nearby[['이름', '거리_km', '거리_점수', '거리_점수_exp']].sort_values('거리_km')
compare.head(20)

## 도전 2 — 큰 카테고리 가중치

In [ ]:
# 대분류 단위 가중치
CATEGORY_WEIGHTS = {
    '문화·학습': 0.4,
    '먹거리':   0.4,
    '여가':     0.2,
}

nearby['대분류_가중치'] = nearby['대분류'].map(CATEGORY_WEIGHTS).fillna(0.2)
nearby['총점_v2'] = nearby['거리_점수'] * nearby['대분류_가중치']

top10_v2 = nearby.sort_values('총점_v2', ascending=False).head(10)
top10_v2[['이름', '종류', '대분류', '거리_km', '총점_v2']].round(2)

## 도전 3 — Heatmap 추가

In [ ]:
from folium.plugins import HeatMap

m_heat = folium.Map(location=[BASE_LAT, BASE_LON], zoom_start=14)
folium.Marker([BASE_LAT, BASE_LON], icon=folium.Icon(color='black', icon='star')).add_to(m_heat)

# 시설 밀도를 heatmap으로
heat_data = [[row['lat'], row['lon']] for _, row in nearby.iterrows()]
HeatMap(heat_data, radius=15).add_to(m_heat)

m_heat.save(DATA_DIR + 'heatmap.html')
m_heat

## 도전 4 — Excel로 export

In [ ]:
with pd.ExcelWriter(DATA_DIR + '추천결과.xlsx') as writer:
    top10[['이름', '종류', '거리_km', '총점']].round(2).to_excel(writer, sheet_name='TOP10', index=False)
    nearby[['이름', '종류', '거리_km', '총점']].round(2).to_excel(writer, sheet_name='전체', index=False)
    pd.DataFrame([TYPE_WEIGHTS]).to_excel(writer, sheet_name='가중치', index=False)
    pd.DataFrame({'부대좌표': [f'({BASE_LAT}, {BASE_LON})'], '인사이트': [INSIGHT]}).to_excel(writer, sheet_name='메타', index=False)
print('✅ Excel 저장 완료')

---
# 🎬 수고하셨습니다!

**내일 가져올 것**
1. `nearby.csv` (시설 통합 + 점수)
2. 본인 가중치 + TOP 10 결과
3. `recommendations.html` (folium 지도)
4. Colab 노트북 (정리된 버전)

> *지도 위에 점이 찍히는 순간, 데이터가 살아납니다.*
